# Escenarios de crecimiento poblacional y padronal — Sonora

Esta libreta construye **escenarios** (no pronósticos oficiales) de
crecimiento de población y de padrón electoral para los municipios de
Sonora y sus 21 distritos locales.

Fuentes (artefactos de `00_preparacion_datos`):

| Archivo | Contenido |
|---|---|
| `conapo_sonora.csv` | Proyecciones CONAPO por municipio, sexo y año (1990–2040), con población 18+ |
| `censo_inegi_municipios.csv` | Censos INEGI 2010 y 2020 por municipio (total, sexo, 18+) |
| `secciones_{año}.gpkg` | Padrón (`pad`) y lista nominal (`ln`) por sección, 2019–2025 |
| `conteos_sonora.csv.gz` | Conteo censal electoral INE 2009–2024 (`ln` por sección) |
| `seccion_distrito_por_anio.csv` | Asignación sección → distrito local por año |

**Metodología.** El escenario *base* poblacional es la trayectoria
oficial de CONAPO. Los escenarios *bajo* y *alto* reproyectan el último
dato observado con la tasa compuesta anual (CAGR) observada, modulada
por un factor ±δ configurable. Para el padrón se comparan dos drivers:
el crecimiento padronal observado (2019–2025) y el crecimiento
demográfico de la población en edad de votar (CONAPO 18+).

**Advertencia.** Las cifras del padrón INE reflejan depuración
administrativa y reseccionamientos, no solo demografía. Los escenarios
sirven para dimensionar sensibilidades, no para predecir resultados.


In [1]:
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from vothmo.config import FOLDER_DATA

PROCESSED = FOLDER_DATA / "processed"
MAP_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

# ---------- parámetros de escenario (horizonte libre) ----------
HORIZONTE = 2040          # año final de proyección
DELTA = 0.25              # los escenarios bajo/alto modulan el CAGR ±25 %
ANIO_BASE_PAD = 2025      # último año con cartografía/padrón observado

conapo = pd.read_csv(PROCESSED / "conapo_sonora.csv")
inegi = pd.read_csv(PROCESSED / "censo_inegi_municipios.csv")
piv = pd.read_csv(PROCESSED / "seccion_distrito_por_anio.csv")

pad_sec = pd.concat([
    gpd.read_file(PROCESSED / f"secciones_{y}.gpkg", ignore_geometry=True)
      .assign(anio=y)
    for y in MAP_YEARS
], ignore_index=True)[["anio", "seccion", "municipio", "mpiocve", "distrito_l", "pad", "ln"]]

conteos = pd.read_csv(PROCESSED / "conteos_sonora.csv.gz",
                      usecols=["feleccion", "mpiocve", "mpionom",
                               "seccion", "del", "ln"],
                      low_memory=False)
conteos["anio_elec"] = conteos["feleccion"].str[-4:].astype(int)

# La cartografía y los conteos del INE usan su propia clave
# municipal; `mun_ine_inegi.csv` la traduce al catálogo INEGI.
# Las capas secciones_*.gpkg ya traen `mpiocve` y `nom_mun`.
munmap = pd.read_csv(PROCESSED / "mun_ine_inegi.csv")

conapo_state = (conapo.groupby("anio")
                .agg(pob=("pob_total", "sum"), pob18=("pob_18mas", "sum"))
                .reset_index())
print("Padrón por año:\n", pad_sec.groupby("anio")[["pad", "ln"]].sum())
print("\nLN conteo censal electoral:\n", conteos.groupby("anio_elec").ln.sum())


Padrón por año:
           pad       ln
anio                  
2019  2032271  2001522
2020  2108953  2072215
2021  2191210  2187614
2022  2140362  2117611
2023  2221477  2197926
2024  2295226  2290980
2025  2317487  2291606

LN conteo censal electoral:
 anio_elec
2009    1826173
2012    1718637
2015    1967629
2018    2101726
2021    2187666
2024    2291013
Name: ln, dtype: int64


## 1. Series históricas

Dos mediciones del "padrón" conviven en los datos:

- **`pad` / `ln` de la cartografía INE** (2019–2025): padrón electoral y
  lista nominal vigentes al corte cartográfico de cada año.
- **`ln` del conteo censal electoral** (2009–2024): lista nominal al
  cierre de cada proceso electoral (años de elección federal/local).

La población se contrasta con los censos INEGI (2010, 2020) y la serie
continua CONAPO (1990–2040).

In [2]:
pad_anual = pad_sec.groupby("anio")[["pad", "ln"]].sum().reset_index()
ln_conteo = conteos.groupby("anio_elec").ln.sum().reset_index()

fig = go.Figure()
fig.add_scatter(x=pad_anual.anio, y=pad_anual.pad, name="Padrón (cartografía)",
                mode="lines+markers")
fig.add_scatter(x=pad_anual.anio, y=pad_anual.ln, name="LN (cartografía)",
                mode="lines+markers")
fig.add_scatter(x=ln_conteo.anio_elec, y=ln_conteo.ln,
                name="LN (conteo censal electoral)", mode="lines+markers")
fig.update_layout(title="Padrón electoral y lista nominal, Sonora",
                  yaxis_title="personas", height=420)
fig.show()

fig = go.Figure()
fig.add_scatter(x=conapo_state.anio, y=conapo_state.pob,
                name="CONAPO total", mode="lines")
fig.add_scatter(x=conapo_state.anio, y=conapo_state.pob18,
                name="CONAPO 18+", mode="lines")
inegi_state = (inegi.groupby("anio")
               .agg(pob=("pobtot", "sum"), pob18=("p_18ymas", "sum"))
               .reset_index())
fig.add_scatter(x=inegi_state.anio, y=inegi_state.pob, name="INEGI censo",
                mode="markers", marker=dict(size=10, symbol="diamond",
                                            color="black"))
fig.add_scatter(x=inegi_state.anio, y=inegi_state.pob18, name="INEGI 18+",
                mode="markers", marker=dict(size=10, symbol="diamond-open",
                                            color="black"))
fig.update_layout(title="Población de Sonora: INEGI vs CONAPO",
                  yaxis_title="personas", height=420)
fig.show()


## 2. Tasas de crecimiento observadas

Se calcula la tasa anual compuesta (CAGR) de cada magnitud en su ventana
disponible. Estas tasas alimentan los escenarios.

In [3]:
def cagr(v0, v1, n):
    return (v1 / v0) ** (1 / n) - 1

tasas = {
    "Padrón cartográfico 2019–2025": cagr(
        pad_anual.pad.iloc[0], pad_anual.pad.iloc[-1], 2025 - 2019),
    "LN conteo 2009–2024": cagr(
        ln_conteo.ln.iloc[0], ln_conteo.ln.iloc[-1], 2024 - 2009),
    "Población INEGI 2010–2020": cagr(
        inegi[inegi.anio == 2010].pobtot.sum(),
        inegi[inegi.anio == 2020].pobtot.sum(), 10),
    "Población 18+ CONAPO 2010–2020": cagr(
        conapo_state.loc[conapo_state.anio == 2010, "pob18"].iloc[0],
        conapo_state.loc[conapo_state.anio == 2020, "pob18"].iloc[0], 10),
}
pd.Series(tasas, name="cagr").mul(100).round(2).rename("CAGR %").to_frame()


,CAGR %
Padrón cartográfico 2019–2025,2.21
LN conteo 2009–2024,1.52
Población INEGI 2010–2020,1.01
Población 18+ CONAPO 2010–2020,1.80


## 3. Escenarios de población por municipio

- **Base**: trayectoria oficial CONAPO (ya es una proyección).
- **Bajo / alto**: último dato CONAPO "observado" (≤2025) reproyectado
  con el CAGR municipal 2015–2025 modulado por ±`DELTA`.

Esto acota la trayectoria oficial con un abanico de incertidumbre
parametrizable.

In [4]:
conapo_tot = (conapo.groupby(["mpiocve", "nom_mun", "anio"])
              .pob_total.sum().reset_index())

anio_ref = conapo_tot.anio[conapo_tot.anio <= 2025].max()   # último "observado"
hist = conapo_tot[conapo_tot.anio.between(2015, anio_ref)]
cagr_mun = (hist.groupby("mpiocve")
            .apply(lambda d: cagr(d.pob_total.iloc[0], d.pob_total.iloc[-1],
                                  d.anio.iloc[-1] - d.anio.iloc[0]),
                   include_groups=False)
            .rename("cagr_obs"))

base_mun = (conapo_tot[conapo_tot.anio >= anio_ref]
            .rename(columns={"pob_total": "pob"}).assign(escenario="base"))

ref = (conapo_tot[conapo_tot.anio == anio_ref][["mpiocve", "nom_mun", "pob_total"]]
       .merge(cagr_mun, on="mpiocve"))

anios = list(range(anio_ref + 1, HORIZONTE + 1))
alt = pd.concat([
    ref.assign(escenario=nom,
               pob=lambda x: x.pob_total * (1 + x.cagr_obs * fac) ** (a - anio_ref),
               anio=a)
    for nom, fac in [("bajo", 1 - DELTA), ("alto", 1 + DELTA)]
    for a in anios
], ignore_index=True)

esc_pob = pd.concat([base_mun[["mpiocve", "nom_mun", "anio", "pob", "escenario"]],
                     alt[["mpiocve", "nom_mun", "anio", "pob", "escenario"]]],
                    ignore_index=True)

state_esc = esc_pob.groupby(["escenario", "anio"]).pob.sum().reset_index()
fig = px.line(state_esc, x="anio", y="pob", color="escenario",
              color_discrete_map={"base": "#1f77b4", "bajo": "#d62728",
                                  "alto": "#2ca02c"},
              title=f"Población estatal proyectada a {HORIZONTE} "
                    f"(base = CONAPO; bajo/alto = CAGR observado ±{DELTA:.0%})")
fig.add_scatter(x=inegi_state.anio, y=inegi_state.pob, name="INEGI censo",
                mode="markers", marker=dict(size=10, symbol="diamond",
                                            color="black"))
fig.update_layout(height=430, yaxis_title="personas")
fig.show()


Detalle municipal: los **6 municipios más poblados**
(Hermosillo, Cajeme, Nogales… concentran la mayor parte de la dinámica).
Cambia `TOP_N` o filtra por `mpiocve` para explorar otros.

In [5]:
TOP_N = 6
top_mpios = (conapo_tot[conapo_tot.anio == anio_ref]
             .nlargest(TOP_N, "pob_total")[["mpiocve", "nom_mun"]])
sel = esc_pob.merge(top_mpios, on=["mpiocve", "nom_mun"])

fig = px.line(sel, x="anio", y="pob", color="escenario",
              facet_col="nom_mun", facet_col_wrap=3,
              color_discrete_map={"base": "#1f77b4", "bajo": "#d62728",
                                  "alto": "#2ca02c"},
              title="Escenarios poblacionales, municipios más poblados")
fig.update_yaxes(matches=None, title="")
fig.update_layout(height=520)
fig.show()


## 4. Escenarios de padrón electoral

Dos drivers comparados:

1. **Padronal observado**: CAGR del `pad` cartográfico 2019→2025 por
   municipio.
2. **Demográfico**: CAGR de la población 18+ de CONAPO 2015→2025 por
   municipio — hipótesis de que el padrón sigue a la población en edad
   de votar.

Cada driver genera su propio abanico bajo/base/alto. La comparación
entre drivers es el ejercicio más informativo: donde el padrón crece más
que la población 18+ suele haber migración, rezago de depuración o
ambos.

In [6]:
pad_mun = pad_sec.groupby(["anio", "mpiocve"]).pad.sum().reset_index()
cagr_pad = (pad_mun.pivot(index="mpiocve", columns="anio", values="pad")
            .pipe(lambda p: cagr(p[2019], p[2025], 6)).rename("cagr_pad"))

conapo18 = conapo.groupby(["mpiocve", "anio"]).pob_18mas.sum().reset_index()
cagr_18 = (conapo18[conapo18.anio.between(2015, 2025)]
           .pivot(index="mpiocve", columns="anio", values="pob_18mas")
           .pipe(lambda p: cagr(p[2015], p[2025], 10)).rename("cagr_18"))

drivers = (pad_mun[pad_mun.anio == ANIO_BASE_PAD]
           .rename(columns={"pad": "pad_base"}).drop(columns="anio")
           .merge(cagr_pad.reset_index(), on="mpiocve")
           .merge(cagr_18.reset_index(), on="mpiocve")
           .merge(munmap[["mpiocve", "nom_mun"]], on="mpiocve"))

anios_pad = list(range(ANIO_BASE_PAD + 1, HORIZONTE + 1))
esc_pad = pd.concat([
    drivers.assign(anio=a,
                   pad=lambda x: x.pad_base
                   * (1 + x[cagr_col] * fac) ** (a - ANIO_BASE_PAD),
                   driver=driver, escenario=nom)
    for a in anios_pad
    for driver, cagr_col in [("padronal", "cagr_pad"),
                             ("demografico", "cagr_18")]
    for nom, fac in [("bajo", 1 - DELTA), ("base", 1.0), ("alto", 1 + DELTA)]
], ignore_index=True)

state_pad = (esc_pad.groupby(["driver", "escenario", "anio"])
             .pad.sum().reset_index())
fig = px.line(state_pad, x="anio", y="pad", color="driver",
              line_dash="escenario",
              category_orders={"escenario": ["bajo", "base", "alto"]},
              title=f"Padrón proyectado a {HORIZONTE}: driver padronal vs demográfico")
fig.add_scatter(x=pad_anual.anio, y=pad_anual.pad, name="Padrón observado",
                mode="markers", marker=dict(size=9, color="black"))
fig.update_layout(height=450, yaxis_title="personas en el padrón")
fig.show()


## 5. Agregación a distritos locales

El padrón se puede agregar a distrito directamente (las secciones ya
llevan `distrito_l` en la cartografía 2025): el padrón municipal
proyectado se reparte según la proporción del padrón 2025 que cada
distrito representa dentro del municipio. La misma ponderación reparte
la población proyectada — una aproximación documentada, no una cifra
censal.

In [7]:
# pesos distrito dentro de municipio según padrón 2025
w = (pad_sec[pad_sec.anio == ANIO_BASE_PAD]
     .groupby(["mpiocve", "distrito_l"]).pad.sum().reset_index())
w["peso"] = w.pad / w.groupby("mpiocve").pad.transform("sum")

pad_dl = (esc_pad.merge(w[["mpiocve", "distrito_l", "peso"]], on="mpiocve")
          .assign(pad_dl=lambda d: d.pad * d.peso)
          .groupby(["distrito_l", "anio", "driver", "escenario"])
          .pad_dl.sum().reset_index())

pob_dl = (esc_pob.merge(w[["mpiocve", "distrito_l", "peso"]], on="mpiocve")
          .assign(pob_dl=lambda d: d.pob * d.peso)
          .groupby(["distrito_l", "anio", "escenario"]).pob_dl.sum()
          .reset_index())

fig = px.line(pob_dl[pob_dl.escenario == "base"], x="anio", y="pob_dl",
              color="distrito_l",
              title="Población proyectada por distrito local "
                    "(base CONAPO, reparto por peso de padrón 2025)")
fig.update_layout(height=450, yaxis_title="personas")
fig.show()

sel = pad_dl[(pad_dl.driver == "padronal")]
fig = px.line(sel, x="anio", y="pad_dl", color="escenario",
              facet_col="distrito_l", facet_col_wrap=7,
              color_discrete_map={"base": "#1f77b4", "bajo": "#d62728",
                                  "alto": "#2ca02c"},
              title="Padrón proyectado por distrito local (driver padronal)")
fig.update_yaxes(matches=None, title="")
fig.update_layout(height=700)
fig.show()


## 6. Exportación y lectura de resultados

Se guardan las series para reutilizarlas en `04_padron_vs_censo` y en el
dashboard.

In [8]:
esc_pob.to_csv(PROCESSED / "escenarios_poblacion_mun.csv", index=False)
esc_pad[["mpiocve", "nom_mun", "anio", "pad", "driver", "escenario"]].to_csv(
    PROCESSED / "escenarios_padron_mun.csv", index=False)
pob_dl.to_csv(PROCESSED / "escenarios_poblacion_dl.csv", index=False)
pad_dl.to_csv(PROCESSED / "escenarios_padron_dl.csv", index=False)

print("Población estatal 2030 por escenario:")
print(esc_pob[esc_pob.anio == 2030].groupby("escenario").pob.sum().round(0))
print("\nArchivos escritos:",
      sorted(p.name for p in PROCESSED.glob("escenarios_*.csv")))


Población estatal 2030 por escenario:
escenario
alto    3312915.0
bajo    3242359.0
base    3266041.0
Name: pob, dtype: float64

Archivos escritos: ['escenarios_padron_dl.csv', 'escenarios_padron_mun.csv', 'escenarios_poblacion_dl.csv', 'escenarios_poblacion_mun.csv']


## 7. Lectura e incertidumbre

- **El escenario "base" poblacional es CONAPO**, que ya incorpora
  supuestos de fecundidad, mortalidad y migración. Los escenarios
  bajo/alto son variaciones paramétricas alrededor del crecimiento
  reciente, no proyecciones demográficas alternativas completas.
- **El padrón no es una variable demográfica pura**: crece por nuevos
  registros, depuraciones, actualizaciones de domicilio y
  reseccionamientos. Los años de redistritación (p. ej. 2025) pueden
  alterar tasas municipales sin cambio poblacional real.
- **El reparto municipal → distrito usa pesos de padrón 2025**; los
  municipios que atraviesan varios distritos se reparten según su
  composición electoral actual, que puede cambiar en futuras
  distritaciones.
- Para decisiones formales de distritación rigen los criterios del INE
  (población censal, no padrón). Esta libreta es una herramienta de
  exploración.